# Question: What if HCPC Prices Remained the Same?
## OR at minimum, saw a moderate increase in paid/claim YoY
- I will be investigating HCPC codes that saw a significant increase in paid/claim
    - some (T1019) saw as high as 200% increase across 6 years
        - verify
    - what if these increases weren't the case?
        - what if the costs stayed (relatively) level compared to 2018?
    - what would the difference in paid be?
        - which practices see significant savings amounts?
- flip-side, which codes have remained the same (or similar)?
- Data source: https://opendata.hhs.gov/datasets/medicaid-provider-spending/
## Accompaniment: https://www.youtube.com/watch?v=ocAf1rK3fxE&list=RDZgruBu-puKE&index=3

In [1]:
import pandas as pd
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_rows', None) # no row limit
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

# below loads the data but takes ~2.5 minutes

In [2]:
### UNCOMMENT IN CASE YOU NEED TO RUN AGAIN ####

dataset = ds.dataset(
    "/Users/luketaylor/Desktop/Pycharm/Home_Healthcare_Spending_Trends/Datasets/medicaid-provider-spending.parquet",
    format="parquet"
)
table = dataset.to_table()
medicaid_provider_spending = table.to_pandas()

# convert claim month to datetime for ease of modification later down the line
medicaid_provider_spending['CLAIM_FROM_MONTH_DATE'] = pd.to_datetime(medicaid_provider_spending['CLAIM_FROM_MONTH'])
medicaid_provider_spending['CLAIM_YEAR'] = medicaid_provider_spending['CLAIM_FROM_MONTH_DATE'].dt.year
    #medicaid_provider_spending['CLAIM_FROM_MONTH'].astype(str).str)[:4]
medicaid_provider_spending['CLAIMS_PER_BENEFICIARY'] = medicaid_provider_spending['TOTAL_CLAIMS'] / medicaid_provider_spending['TOTAL_UNIQUE_BENEFICIARIES']
medicaid_provider_spending['PAID_PER_CLAIM'] = medicaid_provider_spending['TOTAL_PAID'] / medicaid_provider_spending['TOTAL_CLAIMS']
medicaid_provider_spending.head()

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CLAIM_FROM_MONTH_DATE,CLAIM_YEAR,CLAIMS_PER_BENEFICIARY,PAID_PER_CLAIM
0,1376609297,1376609297,T1019,2024-07,39765,1205701,1.188877e+08,2024-07-01,2024,30.320659,98.604609
1,1376609297,1376609297,T1019,2024-08,39677,1152534,1.155611e+08,2024-08-01,2024,29.047912,100.266948
2,1376609297,1376609297,T1019,2024-05,39678,1157235,1.128233e+08,2024-05-01,2024,29.165659,97.493815
3,1376609297,1376609297,T1019,2024-06,39834,1164582,1.114492e+08,2024-06-01,2024,29.235879,95.698863
4,1376609297,1376609297,T1019,2024-09,39527,1099808,1.111998e+08,2024-09-01,2024,27.824221,101.108405


# plan of attack:
- identify codes with significant spend that saw a significant increase in paid/claim YoY
- look for top 50(?) codes to help with query-ability
- look in aggregate, which codes had the highest increase in paid/claim
- identify specific practices that saw increases in paid/claim

# 1) Identify High-Spend Codes
- right now limiting to the top 50 based on total paid

In [3]:
hcpc_code_spend = medicaid_provider_spending.groupby(['HCPCS_CODE']).agg(
    BENEFICIARY_COUNT=('TOTAL_UNIQUE_BENEFICIARIES', 'sum'),
    CLAIM_COUNT=('TOTAL_CLAIMS', 'sum'),
    PAID_AMOUNT=('TOTAL_PAID', 'sum'),
    PRACTITIONER_COUNT=('BILLING_PROVIDER_NPI_NUM', 'nunique'),
    PRACTICE_COUNT=('SERVICING_PROVIDER_NPI_NUM', 'nunique'),
)

hcpc_code_spend.reset_index(inplace=True)
hcpc_code_spend['PAID_PER_CLAIM'] = hcpc_code_spend['PAID_AMOUNT'] / hcpc_code_spend['CLAIM_COUNT']
hcpc_code_spend['PAID_PER_BENEFICIARY'] = hcpc_code_spend['PAID_AMOUNT'] / hcpc_code_spend['BENEFICIARY_COUNT']


In [4]:
top_50_codes = hcpc_code_spend.sort_values(by='PAID_AMOUNT', ascending=False).head(50)
# display(top_50_codes)
# top_50_codes.head()

top_50_codes_list = top_50_codes.HCPCS_CODE.unique().tolist()
top_50_codes_set = set(top_50_codes_list) # improved performance later down the road
print(top_50_codes_list)

['T1019', 'T1015', 'T2016', '99213', 'S5125', '99214', '99284', 'H2016', '99283', 'H2015', '99285', '90837', 'S5102', '90834', 'T2021', 'H2017', 'T1017', 'T1020', '90999', 'A0427', '92507', 'H2019', 'T2033', 'T1000', 'H2014', 'H0004', 'S5140', 'H0020', '97530', 'A0429', 'H0019', 'T1040', '99509', '00003', 'T2023', 'T1016', '97153', '97110', 'S9124', 'S5130', 'H2036', 'T2031', 'H0036', 'G0463', 'S5126', 'H0018', 'T2046', 'U0003', 'A0100', 'H2022']


# filter Medicaid utilization data to the top 50 codes by spend
- reduces query time associated due to the sheer reduction in rows
- takes 3 minutes to run

In [5]:
top_50_codes_all_util = top_50_codes_all_util = medicaid_provider_spending.loc[
    medicaid_provider_spending.HCPCS_CODE.isin(top_50_codes_set)
].copy()

top_50_codes_all_util.head()

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CLAIM_FROM_MONTH_DATE,CLAIM_YEAR,CLAIMS_PER_BENEFICIARY,PAID_PER_CLAIM
0,1376609297,1376609297,T1019,2024-07,39765,1205701,1.188877e+08,2024-07-01,2024,30.320659,98.604609
1,1376609297,1376609297,T1019,2024-08,39677,1152534,1.155611e+08,2024-08-01,2024,29.047912,100.266948
2,1376609297,1376609297,T1019,2024-05,39678,1157235,1.128233e+08,2024-05-01,2024,29.165659,97.493815
3,1376609297,1376609297,T1019,2024-06,39834,1164582,1.114492e+08,2024-06-01,2024,29.235879,95.698863
4,1376609297,1376609297,T1019,2024-09,39527,1099808,1.111998e+08,2024-09-01,2024,27.824221,101.108405


# Aggregate HCPC Utilization by the newly filtered DF
groupby options:
- single groupby
- df.groupby(['col_1', 'col_2']).agg({
    'col_3': 'sum',
    'col_4': 'mean'
})


- groupby multiple columns
df.groupby(['col_1', 'col_2']).agg({
    'col_3': ['sum', 'count'],
    'col_4': ['mean', 'max']
})
# takes ~30 seconds to run

In [6]:
monthly_aggregations_by_hcpc = top_50_codes_all_util.groupby(['HCPCS_CODE', 'CLAIM_FROM_MONTH']).agg(
    BENEFICIARY_COUNT=('TOTAL_UNIQUE_BENEFICIARIES', 'sum'),
    CLAIM_COUNT=('TOTAL_CLAIMS', 'sum'),
    PAID_AMOUNT=('TOTAL_PAID', 'sum'),
    PRACTITIONER_COUNT=('BILLING_PROVIDER_NPI_NUM', 'nunique'),
    PRACTICE_COUNT=('SERVICING_PROVIDER_NPI_NUM', 'nunique'),
)

monthly_aggregations_by_hcpc.reset_index(inplace=True)

monthly_aggregations_by_hcpc['CLAIM_FROM_MONTH_DATE'] = pd.to_datetime(monthly_aggregations_by_hcpc['CLAIM_FROM_MONTH'])
monthly_aggregations_by_hcpc['CLAIM_YEAR'] = monthly_aggregations_by_hcpc['CLAIM_FROM_MONTH_DATE'].dt.year

# add in metrics i'd like to break down
- claims per beneficiary
- paid per claim
- paid per beneficary

In [7]:
monthly_aggregations_by_hcpc['CLAIMS_PER_BENEFICIARY'] = monthly_aggregations_by_hcpc['CLAIM_COUNT'] / monthly_aggregations_by_hcpc['BENEFICIARY_COUNT']
monthly_aggregations_by_hcpc['PAID_PER_CLAIM'] = monthly_aggregations_by_hcpc['PAID_AMOUNT'] / monthly_aggregations_by_hcpc['CLAIM_COUNT']
monthly_aggregations_by_hcpc['PAID_PER_BENEFICIARY'] = monthly_aggregations_by_hcpc['PAID_AMOUNT'] / monthly_aggregations_by_hcpc['BENEFICIARY_COUNT']


monthly_aggregations_by_hcpc.sort_values(by=['PAID_AMOUNT','PAID_PER_CLAIM'], ascending=[False, False], inplace=True)

monthly_aggregations_by_hcpc[~monthly_aggregations_by_hcpc.HCPCS_CODE.isin(['T1019','T1015'])].head(100)

,HCPCS_CODE,CLAIM_FROM_MONTH,BENEFICIARY_COUNT,CLAIM_COUNT,PAID_AMOUNT,PRACTITIONER_COUNT,PRACTICE_COUNT,CLAIM_FROM_MONTH_DATE,CLAIM_YEAR,CLAIMS_PER_BENEFICIARY,PAID_PER_CLAIM,PAID_PER_BENEFICIARY
2670,S5125,2024-07,278913,6753420,5.698249e+08,2526,1170,2024-07-01,2024,24.213357,84.375749,2043.020114
722,99213,2023-03,10236627,12300727,5.622849e+08,64085,178558,2023-03-01,2023,1.201639,45.711518,54.928729
2671,S5125,2024-08,284435,6808894,5.618175e+08,2565,1167,2024-08-01,2024,23.938313,82.512303,1975.205330
2668,S5125,2024-05,294172,6956842,5.575038e+08,2761,1374,2024-05-01,2024,23.648892,80.137484,1895.162752
2661,S5125,2023-10,299028,7026626,5.518303e+08,2790,1403,2023-10-01,2023,23.498221,78.534173,1845.413342
816,99214,2024-01,6773160,7912050,5.512468e+08,58174,168212,2024-01-01,2024,1.168148,69.671800,81.386940
3681,T2016,2024-10,54571,935843,5.501297e+08,951,850,2024-10-01,2024,17.149090,587.843987,10080.989536
2673,S5125,2024-10,276433,6614386,5.424061e+08,2623,1341,2024-10-01,2024,23.927628,82.003992,1962.161021
718,99213,2022-11,9682217,11410227,5.390282e+08,61051,168059,2022-11-01,2022,1.178473,47.240797,55.671983
806,99214,2023-03,7410685,8641775,5.383584e+08,61569,175260,2023-03-01,2023,1.166124,62.297200,72.646238


# re-aggregate at the monthly level, look for a HCPC's first & last month with claims billed
- im omitting 2024-11 and 2024-12 due to insufficient claims data
    - looks like the data has runout issues

## also grabbing paid/claim by month and HCPC to enable me to easily grab first + last month's amounts

In [8]:
claims_util_w_runout = monthly_aggregations_by_hcpc[monthly_aggregations_by_hcpc['CLAIM_FROM_MONTH'] <= '2024-09']

start_and_end_months_by_hcpc = claims_util_w_runout.groupby(['HCPCS_CODE','CLAIM_YEAR']).agg(
    FIRST_MONTH=('CLAIM_FROM_MONTH', 'min'),
    LAST_MONTH=('CLAIM_FROM_MONTH', 'max'),
    TOTAL_MONTHS=('CLAIM_FROM_MONTH', 'nunique'),
    YEARLY_CLAIMS=('CLAIM_COUNT','sum'),
)
start_and_end_months_by_hcpc.reset_index(inplace=True)

hcpc_paid_per_claim = monthly_aggregations_by_hcpc[['HCPCS_CODE','CLAIM_FROM_MONTH', 'CLAIM_YEAR','PAID_PER_CLAIM','CLAIM_COUNT']]

start_and_end_months_by_hcpc.head()
# hcpc_paid_per_claim.head()

,HCPCS_CODE,CLAIM_YEAR,FIRST_MONTH,LAST_MONTH,TOTAL_MONTHS,YEARLY_CLAIMS
0,00003,2018,2018-01,2018-12,12,4744653
1,00003,2019,2019-01,2019-12,12,3807981
2,00003,2020,2020-01,2020-12,12,2516866
3,00003,2021,2021-01,2021-12,12,2556097
4,00003,2022,2022-01,2022-12,12,2091562


# get first and last month's paid/claim amount
- get the difference and calculate the annualized + monthly change
- which codes stand out?
- rough & dirty savings calc
- add 5% increase to cost/claim in a given year
    - that * claim count gives us an idea
    - better implementation:
        - monthly-ize the 5% growth rate, apply to the monthly util database
        - if > 1.05% annual, tough time buddy

In [9]:
first_paid_per_claim = pd.merge(
    left=start_and_end_months_by_hcpc,
    right=hcpc_paid_per_claim,
    how='left',
    left_on=['FIRST_MONTH', 'CLAIM_YEAR', 'HCPCS_CODE'],
    right_on=['CLAIM_FROM_MONTH', 'CLAIM_YEAR', 'HCPCS_CODE'],
)


first_paid_per_claim.drop(columns=['CLAIM_FROM_MONTH'], inplace=True)
first_paid_per_claim.rename(columns={
    'PAID_PER_CLAIM':'BEGINNING_PAID_PER_CLAIM',
    # 'CLAIM_COUNT':'BEGINNING_CLAIM_COUNT',
    }, inplace=True)


paid_per_claim_diffs = pd.merge(
    left=first_paid_per_claim,
    right=hcpc_paid_per_claim,
    how='left',
    left_on=['LAST_MONTH', 'CLAIM_YEAR', 'HCPCS_CODE'],
    right_on=['CLAIM_FROM_MONTH', 'CLAIM_YEAR', 'HCPCS_CODE'],
)

paid_per_claim_diffs.drop(columns=['CLAIM_FROM_MONTH'], inplace=True)
paid_per_claim_diffs.rename(columns={
    'PAID_PER_CLAIM':'ENDING_PAID_PER_CLAIM',
    # 'CLAIM_COUNT':'ENDING_CLAIM_COUNT',
    }, inplace=True)


paid_per_claim_diffs['PAID_PER_CLAIM_PCT_CHANGE'] = (paid_per_claim_diffs['ENDING_PAID_PER_CLAIM'] / paid_per_claim_diffs['BEGINNING_PAID_PER_CLAIM'])

paid_per_claim_diffs["MONTHLY_GROWTH_RATE"] = (
    paid_per_claim_diffs["PAID_PER_CLAIM_PCT_CHANGE"]
    ** (1 / paid_per_claim_diffs["TOTAL_MONTHS"])
)

paid_per_claim_diffs["ANNUAL_GROWTH_RATE"] = ( paid_per_claim_diffs["MONTHLY_GROWTH_RATE"] ** 12)
paid_per_claim_diffs['5_PCT_END_OF_YEAR_PPC'] = paid_per_claim_diffs['BEGINNING_PAID_PER_CLAIM'] * 1.05
paid_per_claim_diffs['EST_5_PCT_SAVINGS'] = paid_per_claim_diffs['5_PCT_END_OF_YEAR_PPC'] * paid_per_claim_diffs['YEARLY_CLAIMS']



col_order = [
    'HCPCS_CODE'
    ,'FIRST_MONTH','BEGINNING_PAID_PER_CLAIM'
    # ,'BEGINNING_CLAIM_COUNT'
    ,'LAST_MONTH','ENDING_PAID_PER_CLAIM'
    # ,'ENDING_CLAIM_COUNT'
    ,'PAID_PER_CLAIM_PCT_CHANGE','TOTAL_MONTHS','MONTHLY_GROWTH_RATE','ANNUAL_GROWTH_RATE'
    ,'5_PCT_END_OF_YEAR_PPC', 'EST_5_PCT_SAVINGS'
]

paid_per_claim_diffs[paid_per_claim_diffs['ANNUAL_GROWTH_RATE']>=1.05][col_order]

,HCPCS_CODE,FIRST_MONTH,BEGINNING_PAID_PER_CLAIM,LAST_MONTH,ENDING_PAID_PER_CLAIM,PAID_PER_CLAIM_PCT_CHANGE,TOTAL_MONTHS,MONTHLY_GROWTH_RATE,ANNUAL_GROWTH_RATE,5_PCT_END_OF_YEAR_PPC,EST_5_PCT_SAVINGS
5,00003,2023-01,293.298126,2023-12,310.630711,1.059095,12,1.004796,1.059095,307.963032,7.108652e+08
7,90834,2018-01,61.798758,2018-12,66.023013,1.068355,12,1.005525,1.068355,64.888696,8.038946e+08
8,90834,2019-01,68.621924,2019-12,75.875744,1.105707,12,1.008409,1.105707,72.053021,9.827574e+08
10,90834,2021-01,76.939542,2021-12,81.096267,1.054026,12,1.004394,1.054026,80.786519,1.421943e+09
11,90834,2022-01,82.918498,2022-12,87.586051,1.056291,12,1.004574,1.056291,87.064423,1.550834e+09
12,90834,2023-01,86.741205,2023-12,95.486677,1.100823,12,1.008037,1.100823,91.078265,1.629273e+09
14,90837,2018-01,71.747635,2018-12,77.180511,1.075722,12,1.006101,1.075722,75.335017,9.642649e+08
15,90837,2019-01,78.676394,2019-12,82.753488,1.051821,12,1.004219,1.051821,82.610214,1.168327e+09
19,90837,2023-01,95.034011,2023-12,119.506543,1.257513,12,1.019278,1.257513,99.785711,2.453046e+09
25,90999,2022-01,77.619265,2022-12,85.315747,1.099157,12,1.007910,1.099157,81.500229,1.149948e+09


# 23/50 codes have an annual growth rate of > 5%
## what if we cap this amount at 5%?
- savings logic would be:
- paid/claim not to exceed 5% annually
- easier to just step increase the payment amounts by nmonth
    - if no significant increase, keep as-is

In [10]:
pow(1.05, 1/12)

1.0040741237836484

# mock logic as of a single case to reduce rows + noise
- i want to look at the first month's paid for a given claim code
- and iteratively apply the 5% annualized growth

# starting point:
- monthly_paid_per_claim_increase * last month's paid per claim
- eventually we'll want to start from a baseline - e.g. if we're working with month 4, pulling month 3's PPC to adjust for month 4 but month 3 is 1% higher than month 2, we're off kilter
- starting from the beginning of year (or from the start of data?) progress each month's payment amount by 1.05 ^ (1/12) (5% annualized)

# rank by HCPC and provider to find the first month for sake of getting the first month's paid per claim
## make sure to filter to > $0 paid
# also want sizable claim volume (add later)
# not U0003 when it first began to be billed
code:
- home_health_util_sorted['CLAIMS_MONTHLY_RANK'] = home_health_util_sorted.groupby('CLAIM_FROM_MONTH')['TOTAL_CLAIMS'].rank(method='min', ascending=False)

In [11]:
## don't want zero claims throwing off pricing comparisons
non_zero_paid = top_50_codes_all_util[top_50_codes_all_util['TOTAL_PAID'] > 0]

non_zero_paid['MONTHLY_ORDER'] = non_zero_paid.groupby(['BILLING_PROVIDER_NPI_NUM','SERVICING_PROVIDER_NPI_NUM','HCPCS_CODE'])['CLAIM_FROM_MONTH'].rank(method='min', ascending=True)

non_zero_paid[(non_zero_paid['SERVICING_PROVIDER_NPI_NUM']=='1376609297') & (non_zero_paid['MONTHLY_ORDER'] == 1)].head() ## get the first month of claims for the given provider

first_month_paid_claims = non_zero_paid[(non_zero_paid['MONTHLY_ORDER'] == 1)].copy() ## get the first month of claims for the given provider

first_month_paid_claims.rename(columns={
    'CLAIM_FROM_MONTH_DATE':'FIRST_CLAIM_MONTH',
    'PAID_PER_CLAIM':'FIRST_PAID_PER_CLAIM'
}, inplace=True)

paid_per_claim_by_month_with_first_month = pd.merge(
    left=non_zero_paid,
    right=first_month_paid_claims[['BILLING_PROVIDER_NPI_NUM','SERVICING_PROVIDER_NPI_NUM','HCPCS_CODE','FIRST_CLAIM_MONTH','FIRST_PAID_PER_CLAIM']], ## to reduce resulting noise
    how='left',
    left_on=['BILLING_PROVIDER_NPI_NUM','SERVICING_PROVIDER_NPI_NUM','HCPCS_CODE'],
    right_on=['BILLING_PROVIDER_NPI_NUM','SERVICING_PROVIDER_NPI_NUM','HCPCS_CODE'],
)

paid_per_claim_by_month_with_first_month.head()

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CLAIM_FROM_MONTH_DATE,CLAIM_YEAR,CLAIMS_PER_BENEFICIARY,PAID_PER_CLAIM,MONTHLY_ORDER,FIRST_CLAIM_MONTH,FIRST_PAID_PER_CLAIM
0,1376609297,1376609297,T1019,2024-07,39765,1205701,1.188877e+08,2024-07-01,2024,30.320659,98.604609,79.0,2018-01-01,88.29359
1,1376609297,1376609297,T1019,2024-08,39677,1152534,1.155611e+08,2024-08-01,2024,29.047912,100.266948,80.0,2018-01-01,88.29359
2,1376609297,1376609297,T1019,2024-05,39678,1157235,1.128233e+08,2024-05-01,2024,29.165659,97.493815,77.0,2018-01-01,88.29359
3,1376609297,1376609297,T1019,2024-06,39834,1164582,1.114492e+08,2024-06-01,2024,29.235879,95.698863,78.0,2018-01-01,88.29359
4,1376609297,1376609297,T1019,2024-09,39527,1099808,1.111998e+08,2024-09-01,2024,27.824221,101.108405,81.0,2018-01-01,88.29359


# Calculate Estimated Savings
- right now i have a default annual increase per HCPC of 5% on the paid/claim
- if you get paid 100/claim this year, next year you can earn at max 105
    - continually increasing costs will be the death of us
    - T1019 has doubled in price in 6 years for multiple providers
        - does it make sense to allow a code to double in what its paid out at?
        - shouldn't the government have more say in what they pay out per claim?
            - i understand risk plays a role, but i find it hard to justify doubling the cost per claim in 6 years

In [17]:
paid_per_claim_by_month_with_first_month.dropna(subset=['FIRST_CLAIM_MONTH', 'CLAIM_FROM_MONTH_DATE'], inplace=True)
paid_per_claim_by_month_with_first_month['PCT_CHANGE'] = paid_per_claim_by_month_with_first_month['PAID_PER_CLAIM'] / paid_per_claim_by_month_with_first_month['FIRST_PAID_PER_CLAIM']

paid_per_claim_by_month_with_first_month['MONTHS_SINCE_FIRST_PAID_CLAIM'] = (
    (paid_per_claim_by_month_with_first_month['CLAIM_FROM_MONTH_DATE'].dt.year - paid_per_claim_by_month_with_first_month['FIRST_CLAIM_MONTH'].dt.year) * 12 +
    (paid_per_claim_by_month_with_first_month['CLAIM_FROM_MONTH_DATE'].dt.month - paid_per_claim_by_month_with_first_month['FIRST_CLAIM_MONTH'].dt.month)
)

# assume 5% increase/year
annual_increase = 8

monthly_paid_per_claim_increase = pow(1+annual_increase/100, 1/12)

paid_per_claim_by_month_with_first_month['MONTHLY_CHANGE_RATE'] = paid_per_claim_by_month_with_first_month['PCT_CHANGE'] ** (1/paid_per_claim_by_month_with_first_month['MONTHS_SINCE_FIRST_PAID_CLAIM'])
paid_per_claim_by_month_with_first_month['ANNUALIZED_CHANGE_RATE'] = paid_per_claim_by_month_with_first_month['MONTHLY_CHANGE_RATE'] ** 12
ppc_col_nm = 'PAID_PER_CLAIM_'+str(annual_increase)+'_PCT_MONTHLY_INCREASE'
paid_per_claim_by_month_with_first_month[ppc_col_nm] = paid_per_claim_by_month_with_first_month['FIRST_PAID_PER_CLAIM'] * (monthly_paid_per_claim_increase ** paid_per_claim_by_month_with_first_month['MONTHS_SINCE_FIRST_PAID_CLAIM'])

# if the recalculated paid/claim is less than the given month's paid per claim, there are savings to be had
# covert T/F to use when adjusting savings calcs, if recalculated amount is higher, recoup savings

paid_per_claim_by_month_with_first_month['SAVINGS_INDICATOR'] = (paid_per_claim_by_month_with_first_month['PAID_PER_CLAIM'] > paid_per_claim_by_month_with_first_month['PAID_PER_CLAIM_5_PCT_MONTHLY_INCREASE']).astype(int)

paid_per_claim_by_month_with_first_month['SAVINGS_AMOUNT'] =  paid_per_claim_by_month_with_first_month['SAVINGS_INDICATOR'] * (paid_per_claim_by_month_with_first_month['PAID_PER_CLAIM'] - paid_per_claim_by_month_with_first_month[ppc_col_nm]) * paid_per_claim_by_month_with_first_month['TOTAL_CLAIMS']

In [21]:
paid_per_claim_by_month_with_first_month[paid_per_claim_by_month_with_first_month['SAVINGS_INDICATOR']==1].head(1000)

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID,CLAIM_FROM_MONTH_DATE,CLAIM_YEAR,CLAIMS_PER_BENEFICIARY,...,FIRST_CLAIM_MONTH,FIRST_PAID_PER_CLAIM,PCT_CHANGE,MONTHS_SINCE_FIRST_PAID_CLAIM,MONTHLY_CHANGE_RATE,ANNUALIZED_CHANGE_RATE,PAID_PER_CLAIM_5_PCT_MONTHLY_INCREASE,SAVINGS_INDICATOR,SAVINGS_AMOUNT,PAID_PER_CLAIM_8_PCT_MONTHLY_INCREASE
56,1417262056,1417262056,T1019,2024-09,34818,597907,55056558.57,2024-09-01,2024,17.172353,...,2018-01-01,44.480367,2.070175,80,1.009137,1.115324,61.578677,1,1.063160e+07,74.300784
58,1417262056,1417262056,T1019,2024-08,34690,599234,54781914.50,2024-08-01,2024,17.273969,...,2018-01-01,44.480367,2.055287,79,1.009161,1.115642,61.328816,1,1.054299e+07,73.825787
59,1417262056,1417262056,T1019,2024-07,34337,644264,54761232.80,2024-07-01,2024,18.762967,...,2018-01-01,44.480367,1.910913,78,1.008337,1.104760,61.079969,1,7.502003e+06,73.353826
60,1417262056,1417262056,T1019,2024-10,34486,590591,53809750.65,2024-10-01,2024,17.125529,...,2018-01-01,44.480367,2.048358,81,1.008892,1.112075,61.829557,1,9.646042e+06,74.778838
61,1417262056,1417262056,T1019,2024-05,33732,595379,53401322.33,2024-05-01,2024,17.650273,...,2018-01-01,44.480367,2.016462,76,1.009271,1.117103,60.585299,1,1.028461e+07,72.418937
63,1417262056,1417262056,T1019,2024-06,34074,581629,53250080.13,2024-06-01,2024,17.069584,...,2018-01-01,44.480367,2.058287,77,1.009419,1.119072,60.832131,1,1.085812e+07,72.884883
64,1417262056,1417262056,T1019,2024-01,33875,603803,52946668.11,2024-01-01,2024,17.824443,...,2018-01-01,44.480367,1.971401,72,1.009472,1.119771,59.607946,1,1.032738e+07,70.584752
66,1417262056,1417262056,T1019,2023-11,33363,597700,52417283.45,2023-11-01,2023,17.915056,...,2018-01-01,44.480367,1.971619,70,1.009745,1.123417,59.125198,1,1.076647e+07,69.685155
67,1417262056,1417262056,T1019,2023-10,33142,607495,52416359.25,2023-10-01,2023,18.330065,...,2018-01-01,44.480367,1.939795,69,1.009649,1.122133,58.885292,1,1.035361e+07,69.239665
68,1417262056,1417262056,T1019,2024-04,33504,597705,52277254.98,2024-04-01,2024,17.839810,...,2018-01-01,44.480367,1.966335,75,1.009056,1.114257,60.339469,1,9.268812e+06,71.955970


# Top savings by code

In [19]:
top_savings_by_code = paid_per_claim_by_month_with_first_month.groupby(['HCPCS_CODE']).agg(
    total_savings=('SAVINGS_AMOUNT','sum')
)

top_savings_by_code.sort_values(by='total_savings', ascending=False).head(100)

,total_savings
HCPCS_CODE,
T1019,4.509863e+09
99213,3.301897e+09
99214,3.072360e+09
T1015,2.736077e+09
A0427,2.535499e+09
H2016,2.272946e+09
99284,2.144120e+09
99283,1.929981e+09
A0429,1.851665e+09


# Top Savings by Provider

In [20]:
top_savings_by_billing_npi = paid_per_claim_by_month_with_first_month.groupby(['BILLING_PROVIDER_NPI_NUM']).agg(
    total_savings=('SAVINGS_AMOUNT','sum'),
    unique_codes=('HCPCS_CODE','nunique')
)

top_savings_by_billing_npi.sort_values(by='total_savings', ascending=False).head(100)

,total_savings,unique_codes
BILLING_PROVIDER_NPI_NUM,,
1376554592,1.072105e+09,2
1417262056,4.334084e+08,5
1396049987,3.779293e+08,7
1326111337,2.974571e+08,4
1124494059,2.913499e+08,1
1710176151,2.753578e+08,11
1407897309,2.572113e+08,14
1700090834,2.103901e+08,7
1790835296,1.780733e+08,12
